# Which test tells you what, and what to do when they all reject

The Lagrange-multiplier tests are the oldest tools in spatial econometrics and
the easiest to misread. Run `LM-lag` and `LM-error` on a dataset with any
spatial structure at all and both will usually reject, whichever one is true.
That is not a bug in either test; it is what a score test does when the model it
scores against is misspecified in a *different* direction.

The fix, from Anselin, Bera, Florax and Yoon (1996), is a pair of robust
statistics: each tests one alternative while allowing the other to be present.
This notebook runs the full battery on three datasets generated from three
different truths, so the diagnosis can be compared against the answer.

It closes with standard errors. A test that rejects tells you the model is
wrong; a robust covariance estimator tells you how much to doubt the
coefficients you report anyway. Driscoll-Kraay and Conley handle different
dependence structures, and the difference between them is not cosmetic.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
Pkg.instantiate()

using SpatialEconometrics
using LinearAlgebra, SparseArrays, Statistics, Printf, Random

## 1. Three truths

Three datasets on the same lattice and the same `X`:

| | generated by |
|---|---|
| `y_sar` | `y = rho*W*y + X*beta + e` — a lag of the outcome |
| `y_sem` | `y = X*beta + u`, `u = lambda*W*u + e` — correlated errors |
| `y_sdm` | `y = rho*W*y + X*beta + W*X*theta + e` — both a lag and spatial regressors |

The question each test answers is "is *this* missing", and the only way to judge
a test is to ask it about data where the answer is known.

In [ ]:
rng = MersenneTwister(2026)
L = 14; N = L^2
W  = make_rook_weights(L)
IN = sparse(1.0I, N, N)

RHO, LAM, BETA, THETA = 0.5, 0.5, [1.0, -0.5], [0.6, 0.2]
X = randn(rng, N, 2)

y_sar = (IN - RHO * W) \ (X * BETA .+ randn(rng, N))
y_sem = X * BETA .+ (IN - LAM * W) \ randn(rng, N)
y_sdm = (IN - RHO * W) \ (X * BETA .+ W * X * THETA .+ randn(rng, N))

data = (("SAR truth", y_sar), ("SEM truth", y_sem), ("SDM truth", y_sdm))
@printf("%d units on a %dx%d rook lattice\n", N, L, L)

## 2. The classical pair, and why it over-rejects

`lm_test_lag` scores the OLS residuals against an omitted `W*y`; `lm_test_error`
scores them against an omitted `lambda*W*u`. Each assumes the *other* effect is
absent under its null.

Watch the SAR and SEM columns. The true effect rejects, as it should — and so
does the false one, often just as decisively. A researcher reading only these
two numbers would conclude that both are present in both datasets.

In [ ]:
@printf("%-12s %22s %22s\n", "", "LM-lag", "LM-error")
@printf("%-12s %12s %9s %12s %9s\n", "data", "stat", "p", "stat", "p")
for (lab, yy) in data
    a = lm_test_lag(yy, X, W)
    b = lm_test_error(yy, X, W)
    @printf("%-12s %12.3f %9.4f %12.3f %9.4f\n", lab, a.stat, a.p_value, b.stat, b.p_value)
end
println()
println("Both reject on all three. The classical pair cannot discriminate.")

## 3. The robust pair, which can discriminate

`lm_test_robust_lag` tests for an omitted `W*y` **while allowing error
dependence to be present**, and `lm_test_robust_error` does the reverse. The
correction is a score adjustment, not a different model.

Now the pattern should separate: on SAR data the robust lag test rejects and the
robust error test should not, and on SEM data the reverse. This is the
comparison the battery exists for.

In [ ]:
@printf("%-12s %22s %22s %14s\n", "", "robust LM-lag", "robust LM-error", "diagnosis")
@printf("%-12s %12s %9s %12s %9s\n", "data", "stat", "p", "stat", "p")
for (lab, yy) in data
    a = lm_test_robust_lag(yy, X, W)
    b = lm_test_robust_error(yy, X, W)
    verdict = if a.p_value < 0.05 && b.p_value >= 0.05
        "lag"
    elseif b.p_value < 0.05 && a.p_value >= 0.05
        "error"
    elseif a.p_value < 0.05 && b.p_value < 0.05
        "both reject"
    else
        "neither"
    end
    @printf("%-12s %12.3f %9.4f %12.3f %9.4f %14s\n",
            lab, a.stat, a.p_value, b.stat, b.p_value, verdict)
end

## 4. The Durbin battery, where the same trap reappears

The classical pair has nothing to say about `W*X`. Koley and Bera (2023) extend
the score battery to the Durbin terms, and the package exposes the set:

- `lm_test_sdm` — the joint test of `rho = 0` **and** `theta = 0`
- `lm_test_durbin` — `theta = 0` alone
- `lm_test_robust_lag_durbin` — `rho = 0`, allowing `theta` to be non-zero
- `lm_test_robust_durbin` — `theta = 0`, allowing `rho` to be non-zero
- `lm_test_durbin_terms` — one statistic per regressor, so the question becomes
  *which* variable's spatial lag matters rather than whether any does

The row to study is **SAR truth**, which contains no `W*X` term whatsoever.
`lm_test_durbin` rejects it anyway, decisively. `lm_test_robust_durbin` does
not. This is the same failure as in section 2, one dimension over: a score test
pointed at `theta` while `rho` is non-zero and unmodelled will find `theta`,
because the omitted lag leaves exactly the kind of residual correlation that the
`theta` score responds to.

The per-regressor tests at the end carry a caution of their own. On the SDM data
the true `theta` is `[0.6, 0.2]`, and the *larger* coefficient returns the
*weaker* evidence. Score tests at this sample size are not a reliable guide to
which spatial lag matters, and the ordering of their statistics is not an
ordering of effect sizes.

In [ ]:
@printf("%-12s %12s %12s %16s %14s\n",
        "data", "joint SDM", "Durbin", "robust lag|durbin", "robust durbin")
for (lab, yy) in data
    j  = lm_test_sdm(yy, X, W)
    d  = lm_test_durbin(yy, X, W)
    rl = lm_test_robust_lag_durbin(yy, X, W)
    rd = lm_test_robust_durbin(yy, X, W)
    @printf("%-12s %12.4f %12.4f %16.4f %14.4f\n",
            lab, j.p_value, d.p_value, rl.p_value, rd.p_value)
end
println("(p-values)")
println()

println("per-regressor Durbin tests on the SDM data (truth: theta = ", THETA, "):")
for (j, r) in enumerate(lm_test_durbin_terms(y_sdm, X, W))
    @printf("  W*x%d : stat %8.3f   p %.4f\n", j, r.stat, r.p_value)
end
println("\nand on the SAR data, where no W*X term exists:")
for (j, r) in enumerate(lm_test_durbin_terms(y_sar, X, W))
    @printf("  W*x%d : stat %8.3f   p %.4f\n", j, r.stat, r.p_value)
end

## 5. Standard errors, once you have decided to report anyway

Tests say the model is wrong. They do not say what to report. A robust
covariance estimator answers a narrower question: given dependence of some
assumed shape, what is the standard error worth?

Three shapes, three estimators:

| estimator | assumes |
|---|---|
| `ClusterRobustVCOV` | arbitrary correlation within groups, none between |
| `DriscollKraayVCOV` | arbitrary cross-sectional correlation in each period, serial correlation up to a lag |
| `ConleySpatialHAC` | correlation decaying with distance, up to a cutoff, optionally with time lags |

Driscoll-Kraay needs no coordinates and makes no assumption about *which*
units are correlated. Conley needs coordinates and assumes the correlation
dies with distance. On spatial data the second is the stronger assumption and
the sharper tool when it holds.

The panel below is a SAR panel with two-way fixed effects whose errors are
**heteroskedastic in `x1`** — their standard deviation is `0.3 + 0.5*|x1|`.
Nothing is wrong with the mean structure: `rho`, the fixed effects and both
slopes are all correctly specified, and the estimator recovers them. Only the
error variance departs from what the model's own covariance assumes, and it
departs in a way tied to one particular regressor.

In [ ]:
rngp  = MersenneTwister(707)
Lp    = 10; Np = Lp^2; Tp = 10
Wp    = make_rook_weights(Lp)
coords = zeros(Np, 2)
for r in 1:Lp, c in 1:Lp
    coords[(r - 1) * Lp + c, :] = [Float64(r), Float64(c)]
end

Xp   = randn(rngp, Np * Tp, 2)
mu_i = randn(rngp, Np)
S_lu = lu(sparse(1.0I, Np, Np) - 0.4 * Wp)

yp = zeros(Np * Tp)
for t in 1:Tp
    idx = ((t - 1) * Np + 1):(t * Np)
    het = 0.3 .+ 0.5 .* abs.(Xp[idx, 1])          # heteroskedastic in X
    yp[idx] = S_lu \ (mu_i .+ Xp[idx, :] * [1.0, -0.5] .+ het .* randn(rngp, Np))
end

mp = fit_spatial_panel(SAR(), TwoWayFE(), Wp, Xp, yp, Np, Tp;
                       varnames = ["x1", "x2"])

dk     = DriscollKraayVCOV(kernel = BartlettKernel())
conley = ConleySpatialHAC(coords, 3.0; max_time_lag = 2)

println(robust_coeftable(mp, dk))
println(robust_coeftable(mp, conley))

The *Baseline SE* column is the model's own, *Robust SE* the alternative, and
printing them together is the whole point of `robust_coeftable`: the ratio is
readable without arithmetic. The robust errors here are **smaller** than the
model's on `x2` and `rho`, and larger on `x1`.

A robust standard error coming out smaller is not a reason to relax, and it is
not by itself evidence of a bug — it means the dependence the estimator allows
for is not the dependence that is actually costing you precision. Which of the
two columns to believe is a question with an answer, and section 6 goes and
gets it.

### The same sandwich, built by hand

`robust_vcov` is a convenience over `vcov_driscoll_kraay` and `vcov_conley`,
which take a regressor matrix and a residual vector directly. Calling them that
way is what you need when the model is not one of the package's own fits — a
pooled OLS, say, or a two-stage estimate assembled by hand.

Below, both are applied to a pooled OLS on the same panel. The point is not that
the numbers match the table above — the model differs, so they should not — but
that the low-level entry points are available and take ordinary arrays.

In [ ]:
Z = hcat(ones(Np * Tp), Xp)                       # pooled OLS design
b_ols = Z \ yp
e_ols = yp .- Z * b_ols

V_ols = inv(Z' * Z) .* (sum(abs2, e_ols) / (length(yp) - size(Z, 2)))
V_dk2  = vcov_driscoll_kraay(Z, e_ols, Np, Tp)
V_con2 = vcov_conley(Z, e_ols, Np, Tp, ConleySpatialHAC(coords, 3.0; max_time_lag = 2))

@printf("%-12s %12s %12s %12s %10s %10s\n",
        "coefficient", "estimate", "OLS SE", "DK SE", "Conley SE", "DK / OLS")
for (j, nm) in enumerate(["intercept", "x1", "x2"])
    s0 = sqrt(V_ols[j, j]); s1 = sqrt(V_dk2[j, j]); s2 = sqrt(V_con2[j, j])
    @printf("%-12s %12.5f %12.5f %12.5f %10.5f %10.2fx\n",
            nm, b_ols[j], s0, s1, s2, s1 / s0)
end
println()
println("OLS here ignores both the spatial lag and the dependence, so its")
println("standard errors are the ones to distrust most.")

## 6. Which standard error is right

Everything above is an assertion about a sampling distribution, so it can be
checked against one. Refit the same design 150 times on fresh errors, and the
standard deviation of the 150 estimates *is* the standard error the others are
trying to estimate. The grid is smaller here so the loop stays quick.

Read the `MC sd` column as the truth and the rest as attempts at it.

In [ ]:
Lm = 7; Nm = Lm^2; Tm = 10; Wm = make_rook_weights(Lm)
cm = zeros(Nm, 2)
for r in 1:Lm, c in 1:Lm
    cm[(r - 1) * Lm + c, :] = [Float64(r), Float64(c)]
end
Sm   = lu(sparse(1.0I, Nm, Nm) - 0.4 * Wm)
reps = 150
est = zeros(reps, 3); s_base = zeros(reps, 3)
s_dk = zeros(reps, 3); s_con = zeros(reps, 3)

for r in 1:reps
    rng = MersenneTwister(5000 + r)
    Xr  = randn(rng, Nm * Tm, 2); mur = randn(rng, Nm); yr = zeros(Nm * Tm)
    for t in 1:Tm
        ix  = ((t - 1) * Nm + 1):(t * Nm)
        het = 0.3 .+ 0.5 .* abs.(Xr[ix, 1])
        yr[ix] = Sm \ (mur .+ Xr[ix, :] * [1.0, -0.5] .+ het .* randn(rng, Nm))
    end
    mr = fit_spatial_panel(SAR(), TwoWayFE(), Wm, Xr, yr, Nm, Tm;
                           varnames = ["x1", "x2"])
    tb = robust_coeftable(mr, DriscollKraayVCOV(kernel = BartlettKernel()))
    tc = robust_coeftable(mr, ConleySpatialHAC(cm, 3.0; max_time_lag = 2))
    est[r, :] = tb.coef; s_base[r, :] = tb.se_baseline
    s_dk[r, :] = tb.se_robust; s_con[r, :] = tc.se_robust
end

truth = [1.0, -0.5, 0.4]
@printf("%-6s %8s %9s %9s %10s %9s %9s\n",
        "param", "true", "MC mean", "MC sd", "mean base", "mean DK", "mean Con")
for (j, nm) in enumerate(["x1", "x2", "rho"])
    @printf("%-6s %8.3f %9.4f %9.5f %10.5f %9.5f %9.5f\n",
            nm, truth[j], mean(est[:, j]), std(est[:, j]),
            mean(s_base[:, j]), mean(s_dk[:, j]), mean(s_con[:, j]))
end

## Takeaways

**On choosing a specification.**

1. `lm_test_lag` and `lm_test_error` both reject on all three truths. A
   classical score test is computed under the null that *everything else* is
   correctly specified, so the lag test sees omitted error dependence as
   evidence of a lag, and the error test sees an omitted lag as evidence of
   error dependence. Two rejections are not two findings.
2. `lm_test_robust_lag` and `lm_test_robust_error` discriminate correctly on
   all three. Reach for the robust pair first; read the classical pair only as
   "there is spatial structure of some kind here".
3. The same trap sits in the Durbin dimension. On SAR data, where there is no
   `W*X` term at all, `lm_test_durbin` rejects decisively and
   `lm_test_robust_durbin` does not.
4. `lm_test_durbin_terms` says which regressor's lag is implicated, but at
   this sample size the ordering of its statistics is not an ordering of
   effect sizes: on the SDM data the larger true `theta` returned the weaker
   evidence.

**On standard errors.**

5. The model's own covariance was accurate for `x2` and `rho` and understated
   `x1` by a third. That is not bad luck — the errors were built
   heteroskedastic in `x1`, and a misspecified variance shows up in the
   coefficient it is tied to, not evenly across the table.
6. `ConleySpatialHAC` came closest on `x1`, the coefficient that needed help,
   and ran mildly conservative on the other two. `DriscollKraayVCOV`
   understated `x2` and `rho` more. Neither dominates, and a robust estimator
   is not a free upgrade: it buys protection against the dependence it models
   and pays for it in precision elsewhere.
7. A robust standard error that comes out *smaller* than the model's tells you
   the dependence you allowed for is not what is costing you precision. It is
   not a reason to relax and not a licence to report the smaller number.
8. If a standard error matters to a conclusion, simulate the design. Section 6
   is twenty lines, and it is the only part of this notebook that establishes
   rather than assumes.

**What the tests cannot do.** None of this chooses a model for you. A rejection
is evidence that a restriction is false, not evidence for the alternative that
replaces it, and the best-behaved test here still cannot tell a spatial process
from an omitted variable that happens to be spatially smooth.